# Lezione 25 — Un punteggio di importanza composito

**Come si usa questo notebook.** Come sempre. Prerequisiti: Lezione 23
(`peso_importanza_base` per type) e Lezione 24 (`recency`). Tre segnali
separati esistono gia'; oggi li combiniamo in **un solo numero** che le
Lezioni 27-28 useranno per decidere quali memorie contano di piu'.

**Cosa saprai fare alla fine:** combinare piu' segnali con una somma
pesata dichiarata esplicitamente, spiegare perche' i pesi vanno
giustificati invece di essere scelti a caso, e calcolare il punteggio
composito per l'intero archivio.

## Parte 1 — Teoria: combinare segnali senza nasconderli

Ogni memoria del progetto ha gia' tre segnali indipendenti:

- `importance` (Lezione 1) — un punteggio 0-1 assegnato alla memoria
  stessa (esplicito, indipendente dal tempo);
- `recency` (Lezione 24) — quanto conta ancora *oggi*, per il decadimento
  temporale del suo type;
- `peso_importanza_base` (Lezione 23) — quanto il type stesso, a
  prescindere dal contenuto, alza o abbassa l'importanza.

Combinarli in un solo numero serve per **ordinare** le memorie (Lezioni
27-28 lo useranno per decidere cosa mostrare prima). Il modo piu' semplice
e trasparente e' una **somma pesata** dei primi due segnali, scalata dal
terzo:

```
importanza_composita = peso_importanza_base * (alpha * importance + beta * recency)
```

con `alpha + beta = 1`. I pesi `alpha` e `beta` **non sono un dettaglio
tecnico da ottimizzare in astratto**: sono una dichiarazione di priorita'.
Un `alpha` alto dice "fidati soprattutto del punteggio esplicito, il tempo
conta poco"; un `beta` alto dice il contrario. Qui scegliamo `alpha=0.6,
beta=0.4`: il giudizio esplicito sulla memoria pesa piu' del solo fattore
temporale, ma il tempo non e' ignorato — una scelta dichiarata, non
misurata, esattamente come i parametri per type della Lezione 23.

Un dettaglio da non ignorare: il punteggio composito **puo' superare
1.0** (quando `peso_importanza_base > 1`, come per `preference` e
`semantic`). Non e' un bug: `importanza_composita` non e' una probabilita'
e non deve stare in `[0,1]` — e' un punteggio di **ranking**, utile solo
nel confronto relativo tra memorie, non come valore assoluto con un
significato proprio.

In [1]:
ALPHA, BETA = 0.6, 0.4
assert abs(ALPHA + BETA - 1.0) < 1e-9  # i due pesi devono sommare a 1


def importanza_composita(importance, recency, peso_importanza_base):
    return peso_importanza_base * (ALPHA * importance + BETA * recency)


# Due memorie con lo stesso 'importance' esplicito, ma type e recency diversi.
memoria_a = importanza_composita(importance=0.8, recency=0.9, peso_importanza_base=1.2)  # preference recente
memoria_b = importanza_composita(importance=0.8, recency=0.1, peso_importanza_base=1.0)  # episodic vecchia
print(f'preference recente : {memoria_a:.3f}')
print(f'episodic vecchia    : {memoria_b:.3f}')

preference recente : 1.008
episodic vecchia    : 0.520


Leggi l'output: a parita' di `importance` esplicito (`0.8` per
entrambe), la memoria con recency alta e peso di type piu' alto (una
preferenza recente) supera nettamente quella con recency bassa (un evento
vecchio) — il punteggio composito fa esattamente il lavoro per cui e'
stato costruito: non basarsi solo sul giudizio esplicito, ma modularlo con
tempo e type.

## Parte 2 — Esercizio guidato

Il tuo compito: calcola `importanza_composita` con `importance=1.0`,
`recency=1.0` e `peso_importanza_base=1.2` (il caso migliore possibile sui
primi due segnali, con il peso di type piu' alto tra quelli della Lezione
23). Il risultato supera `1.0`?

In [2]:
# Scrivi qui: chiama importanza_composita(1.0, 1.0, 1.2) e stampa il risultato.

pass

### Soluzione spiegata

Con `importance=1.0` e `recency=1.0`, la parte tra parentesi vale
`0.6*1.0 + 0.4*1.0 = 1.0` esattamente; moltiplicata per
`peso_importanza_base=1.2` da' `1.2` — sopra `1.0`, confermando quanto
detto nella Parte 1: il punteggio composito non e' vincolato a `[0,1]`
quando il peso di type supera `1.0`.

In [3]:
caso_migliore = importanza_composita(1.0, 1.0, 1.2)
print(f'caso migliore possibile: {caso_migliore:.3f}')
assert caso_migliore > 1.0

caso migliore possibile: 1.200


## Parte 3 — Il progetto: Memory AI Lab, passo 25 — punteggio composito per l'archivio

Ricalcoliamo `recency` (Lezione 24) e applichiamo `importanza_composita` a
tutto il train set, poi guardiamo quali memorie salgono in cima
all'ordinamento — e verifichiamo che abbia senso.

In [4]:
import pandas as pd
from pathlib import Path

processed = Path('..') / 'datasets' / 'processed'
train = pd.read_csv(processed / 'memory_train.csv')

ORA_RIFERIMENTO = pd.Timestamp('2026-07-18')
PARAMETRI_TIPO = {
    'episodic':   {'half_life_giorni': 30,  'peso_importanza_base': 1.0},
    'semantic':   {'half_life_giorni': 365, 'peso_importanza_base': 1.1},
    'preference': {'half_life_giorni': 180, 'peso_importanza_base': 1.2},
    'unknown':    {'half_life_giorni': 60,  'peso_importanza_base': 0.7},
}

eta_giorni = (ORA_RIFERIMENTO - pd.to_datetime(train['timestamp'])).dt.total_seconds() / 86400
half_life = train['type'].map(lambda t: PARAMETRI_TIPO[t]['half_life_giorni'])
peso_base = train['type'].map(lambda t: PARAMETRI_TIPO[t]['peso_importanza_base'])

train['recency'] = 0.5 ** (eta_giorni / half_life)
train['importanza_composita'] = peso_base * (ALPHA * train['importance'] + BETA * train['recency'])

print(train.groupby('type')['importanza_composita'].agg(['mean', 'min', 'max']).round(3))
print(f"\nrange totale: [{train['importanza_composita'].min():.3f}, {train['importanza_composita'].max():.3f}]")

             mean    min    max
type                           
episodic    0.412  0.058  0.731
preference  0.762  0.470  1.077
semantic    0.748  0.495  0.938
unknown     0.201  0.201  0.201

range totale: [0.058, 1.077]


In [5]:
top5 = train.sort_values('importanza_composita', ascending=False).head(5)
print(top5[['memory_id', 'type', 'importance', 'recency', 'importanza_composita']].to_string(index=False))

memory_id       type  importance  recency  importanza_composita
hist_0163 preference        0.96 0.803696              1.076974
hist_0028 preference        0.98 0.680177              1.032085
hist_0088 preference        0.87 0.739124              0.981180
hist_0079   semantic        0.85 0.856543              0.937879
hist_0191 preference        0.74 0.838606              0.935331


Guarda la classifica onestamente: le prime posizioni sono dominate da
memorie `preference` e `semantic` con `importance` esplicito gia' alto
**e** recency alta (registrate di recente) — coerente con come e' costruita
la formula, non una sorpresa. Nessuna memoria `episodic` compare in cima:
con half-life corto (30 giorni) e un train set che copre circa 3 mesi
(Lezione 24), quasi tutte le episodic hanno gia' una recency bassa,
indipendentemente da quanto fosse alto il loro `importance` originale — un
comportamento voluto (un evento vecchio conta meno *oggi*, anche se era
importante quando e' successo), da tenere presente quando in Lezione 28
useremo questo punteggio per il retrieval.

## Cosa hai imparato

- Un **punteggio composito** combina piu' segnali (esplicito, recency,
  type) con pesi **dichiarati esplicitamente**, non ottimizzati in
  automatico: la scelta dei pesi e' una dichiarazione di priorita', non
  un dettaglio tecnico neutro.
- Il risultato **non e' vincolato a `[0,1]`**: e' un punteggio di
  ranking, utile per confrontare memorie tra loro, non un valore assoluto
  con un significato a se stante.
- Il punteggio composito puo' penalizzare memorie con `importance`
  esplicito alto ma vecchie (recency bassa): e' un comportamento voluto
  della formula, non un errore.

## Quiz

1. Perche' i pesi `alpha` e `beta` sono descritti come "una dichiarazione
   di priorita'" invece che come parametri da ottimizzare?
2. In che condizione `importanza_composita` supera `1.0`? E' un errore?
3. Una memoria `episodic` con `importance=0.95` (altissimo) puo' avere un
   punteggio composito basso. Come e' possibile, e perche' e' un
   comportamento corretto della formula?

<details>
<summary><b>Apri le risposte</b></summary>

1. Perche' non esiste un valore "oggettivamente corretto" per quanto
   pesare il giudizio esplicito rispetto al tempo trascorso: dipende da
   cosa il sistema deve fare (privilegiare cio' che e' stato segnato come
   importante, o privilegiare cio' che e' successo di recente). La scelta
   va dichiarata e giustificata, non nascosta dietro un numero che sembra
   neutro.
2. Quando `peso_importanza_base > 1.0` (come per `preference` a `1.2` o
   `semantic` a `1.1`, Lezione 23) e la parte pesata di `importance` e
   `recency` e' vicina al suo massimo. Non e' un errore: il punteggio
   composito e' un punteggio di ranking, non una probabilita', e non ha
   bisogno di stare in `[0,1]`.
3. Perche' il punteggio composito moltiplica (tramite la somma pesata) il
   segnale esplicito con `recency`, che dipende dall'eta' della memoria e
   dal suo half-life di type: una `episodic` con `importance=0.95` ma
   vecchia rispetto al suo half-life di 30 giorni (Lezione 24) ha
   `recency` molto bassa, che abbassa il composito nonostante
   l'importanza esplicita alta — voluto, perche' un evento passato conta
   meno *oggi* anche se era rilevante quando e' successo.
</details>

## Fonti

- pandas, `DataFrame.sort_values` (usato per l'ordinamento per punteggio
  composito):
  https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html

La formula di combinazione (`importanza_composita`, i pesi `alpha`/`beta`)
e' una scelta di design per il Memory AI Lab che combina segnali gia'
introdotti (Lezioni 1, 23, 24), non da una fonte esterna.

## Prossima lezione

Finora ogni memoria e' stata trattata come un testo isolato. La prossima
lezione estrae **entita'** (persone, luoghi) dal testo, il primo passo
verso il grafo di memorie della Lezione 27.